## Exercise 3: Install Libraries Notebook-Scoped

At HealthBridge Analytics, different teams sometimes need different versions of the same library. Rather than creating a new cluster for every variation, you can install libraries **notebook-scoped** — they apply only to the current notebook session and do not affect other users or notebooks on the same compute resource.

In this exercise, you install the `faker` library notebook-scoped and verify it works correctly.

[faker](https://faker.readthedocs.io/) is a Python library that generates realistic fake data — such as names, addresses, phone numbers, dates, and more — for use in testing, prototyping, and seeding databases. It supports many locales and data categories, making it easy to produce large volumes of plausible synthetic data without using real personal information.

### Task 3.1 — Install the `faker` library notebook-scoped

Your team relies on `faker` to generate synthetic patient records for pipeline testing. Install it as a notebook-scoped library so the package is available to this notebook only and doesn't affect other users sharing the same compute.

> 🤖 **Genie Code tip:** Open Genie Code panel (click the ![assistant-icon](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Allfiles/media/genie-code.svg) icon on the cell) and use the prompt below to get started:
> *"How do I install a Python package notebook-scoped in Databricks using a magic command?"*

**Hint:** Use a `%pip` magic command to install the package. Pinning an exact version (e.g., `faker==40.8.0`) is a good practice for reproducibility.

In [0]:
# TODO: Install faker==40.8.0 notebook-scoped
!pip install faker==40.8.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Task 3.2 — Verify the installation

Before using `faker` in a real pipeline, confirm it was installed correctly by:

1. Importing the `Faker` class from the `faker` library.
2. Creating a `Faker` instance.
3. Printing a randomly generated **full name** and **date of birth**.

> 🤖 **Genie Code tip:**
> *"Show me a Python example of generating a random name and date of birth using the Faker library."*

In [0]:
# Import Faker, create an instance, and print a generated name and date of birth
from faker import Faker

fake = Faker()

print("Name:", fake.name())
print("Date of Birth:", fake.date_of_birth())

Name: Jennifer Conley
Date of Birth: 1992-04-17


## Exercise 4: Generate and Analyze Synthetic Patient Data

Now that `faker` is installed, put it to work. Your team needs a synthetic dataset of patient admission records to test a new data pipeline. The data must look realistic enough to validate transformations and aggregations, but must contain **no real patient information** to comply with HIPAA policies.

In this exercise, you generate a synthetic dataset and run a basic analysis on it.

### Task 4.1 — Generate a synthetic patient admissions DataFrame

Use `faker` and the Spark session (`spark`) to create a **Spark DataFrame** with **100 synthetic patient admission records**. Each record must include:

| Column | Description |
|---|---|
| `patient_id` | Integer, 1 to 100 |
| `full_name` | Randomly generated full name |
| `date_of_birth` | Random date between `1940-01-01` and `2005-12-31`, as a string |
| `admission_date` | Random date between `2023-01-01` and `2025-12-31`, as a string |
| `diagnosis_code` | One of: `I21`, `J18`, `E11`, `K80`, `N39` (randomly chosen) |

Display the first 10 rows of the resulting DataFrame.

> 🤖 **Genie Code tip:**
> *"How do I generate a list of Python dictionaries using the Faker library, then create a PySpark DataFrame from that list?"*
>
> **Hint:** Use `faker.date_of_birth()` with `minimum_age` and `maximum_age` parameters, or `faker.date_between()` with `start_date` and `end_date`. Use `random.choice()` for the diagnosis code.

In [0]:
%pip install faker==40.8.0
# Generate 100 synthetic patient records and create a Spark DataFrame
from faker import Faker
import random
from datetime import date

fake = Faker()
Faker.seed(42)
random.seed(42)

diagnosis_codes = ["I21", "J18", "E11", "K80", "N39"]

records = [
    {
        "patient_id": i,
        "full_name": fake.name(),
        "date_of_birth": str(fake.date_between(start_date=date(1940, 1, 1), end_date=date(2005, 12, 31))),
        "admission_date": str(fake.date_between(start_date=date(2023, 1, 1), end_date=date(2025, 12, 31))),
        "diagnosis_code": random.choice(diagnosis_codes),
    }
    for i in range(1, 101)
]

patients_df = spark.createDataFrame(records)

# Display the first 10 rows
display(patients_df.limit(10))

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


admission_date,date_of_birth,diagnosis_code,full_name,patient_id
2025-03-17,1954-09-24,I21,Allison Hill,1
2023-02-02,1967-11-05,I21,Stephanie Miller,2
2024-12-12,1953-02-14,E11,Jonathan Johnson,3
2023-01-08,1993-06-03,J18,Connie Lawrence,4
2025-11-14,1950-04-05,J18,Melissa Peterson,5
2024-10-23,1995-12-07,J18,Brandon Davis,6
2024-02-19,2004-03-22,I21,Monica Herrera,7
2024-09-24,1996-11-14,N39,Sandra Montgomery,8
2023-03-29,1959-02-05,I21,Amber Perez,9
2024-02-04,1981-12-14,N39,Brian Ramirez,10


### Task 4.2 — Analyze admissions by diagnosis code

The clinical informatics team wants to know which diagnosis codes appear most frequently in the admission data. Using the Spark DataFrame you created in Task 4.1:

1. Count the number of admissions per `diagnosis_code`.
2. Order the results from **most to least** admissions.
3. Display the result.

> 🤖 **Genie Code tip:**
> *"How do I group by a column, count occurrences, and sort the result in descending order using PySpark DataFrame API?"*

In [0]:
# Count admissions per diagnosis_code, ordered from most to least
from pyspark.sql import functions as F

admissions_by_diagnosis = patients_df.groupBy("diagnosis_code").count().orderBy(F.col("count").desc())
display(admissions_by_diagnosis)

diagnosis_code,count
J18,24
I21,23
E11,22
N39,16
K80,15


## Spark UI Sample Code

The cells below generate workloads you can inspect in the **Spark UI** (compute resource > **Spark UI**, or the **See performance** / **Spark Jobs** link under a cell's output).

| Sample | Where to look in the Spark UI |
|---|---|
| 1. Job descriptions and job groups | **Jobs** tab |
| 2. Narrow vs. wide transformations (shuffle) | **Jobs** > job > **DAG Visualization**, **Stages** tab |
| 3. Partitions and tasks | **Stages** > stage > task list / **Event Timeline** |
| 4. Caching | **Storage** tab |
| 5. Broadcast vs. sort-merge join | **SQL / DataFrame** tab (query plan) |
| 6. Data skew | **Stages** > **Summary Metrics** (max vs. median task duration) |
| 7. Adaptive Query Execution (AQE) | **SQL / DataFrame** tab (`AdaptiveSparkPlan`, `AQEShuffleRead`) |
| 8. Shuffle spill | **Stages** > *Spill (Memory)* / *Spill (Disk)* columns |
| 9. Configuration and executors | **Environment** and **Executors** tabs |

> These samples use `spark.sparkContext` for job descriptions/groups and the UI URL. That works on classic compute with **Dedicated** access mode; on Standard (shared) access mode or serverless, skip those lines.

In [0]:
# Sample 1: Label jobs so they are easy to find in the Spark UI > Jobs tab
spark.addTag("healthbridge_demo")
spark.addTag("sample_1_simple_count")

df = spark.range(0, 10_000_000)
print("Row count:", df.count())  # Action -> triggers a job

spark.removeTag("sample_1_simple_count")  # Reset the tag for later jobs

Row count: 10000000


In [0]:
# Sample 2: Narrow transformations (filter, withColumn) stay in one stage;
# a wide transformation (groupBy) introduces a shuffle and a new stage.
from pyspark.sql import functions as F

spark.conf.set("spark.job.description", "Sample 2 - narrow vs wide transformations")

narrow_df = (
    spark.range(0, 20_000_000)
    .withColumn("bucket", F.col("id") % 100)   # narrow
    .filter(F.col("id") % 2 == 0)              # narrow
)

wide_df = narrow_df.groupBy("bucket").agg(F.count("*").alias("cnt"))  # wide -> shuffle

display(wide_df.orderBy("bucket"))
# In the UI: open the job > DAG Visualization and look for the Exchange (shuffle) boundary.
# In the Stages tab, compare 'Shuffle Write' and 'Shuffle Read' between the stages.

bucket,cnt
0,200000
2,200000
4,200000
6,200000
8,200000
10,200000
12,200000
14,200000
16,200000
18,200000


In [0]:
# Sample 3: The number of partitions determines the number of tasks in a stage
from pyspark.sql import functions as F

spark.conf.set("spark.job.description", "Sample 3 - partitions and tasks")

base_df = spark.range(0, 10_000_000)
print("Default partitions:", base_df.select(F.spark_partition_id()).distinct().count())

few_parts = base_df.repartition(4)
many_parts = base_df.repartition(64)

few_parts.selectExpr("sum(id)").collect()   # Stage with 4 tasks
many_parts.selectExpr("sum(id)").collect()  # Stage with 64 tasks

# In the UI: Stages tab > compare the 'Tasks' column and open each stage's Event Timeline.
# To open the Spark UI:
#   1. Click the cluster/compute name in the top-right of the notebook.
#   2. Select "Spark UI" (or go to Compute > your cluster > Spark UI).
#   3. Navigate to the **Stages** tab.
#   4. Compare the two stages: one should show 4 tasks, the other 64 tasks.
#   5. Click into each stage and open the **Event Timeline** to see task parallelism.

Default partitions: 4


[Row(sum(id)=49999995000000)]

In [0]:
# Sample 4: Cached data appears in the Spark UI > Storage tab
spark.conf.set("spark.job.description", "Sample 4 - caching")

cached_df = spark.range(0, 5_000_000).withColumn("value", F.rand(seed=42))
cached_df.cache()
cached_df.count()  # First action materializes the cache
cached_df.count()  # Second action reads from cache (check the faster job duration)

# In the UI: Storage tab shows the cached RDD, its storage level, size in memory and fraction cached.
# Free the memory when done:
cached_df.unpersist()

DataFrame[id: bigint, value: double]

In [0]:
# Sample 5: Compare join strategies in the Spark UI > SQL / DataFrame tab
spark.conf.set("spark.job.description", "Sample 5 - join strategies")

facts = spark.range(0, 10_000_000).withColumn("dim_id", F.col("id") % 1000)
dims = spark.range(0, 1000).withColumnRenamed("id", "dim_id").withColumn("label", F.concat(F.lit("dim_"), F.col("dim_id")))

# Broadcast hash join: small table is sent to every executor, no shuffle of the large table
broadcast_join = facts.join(F.broadcast(dims), "dim_id")
broadcast_join.explain()
broadcast_join.count()

# Sort-merge join: hint forces both sides to be shuffled and sorted
smj_join = facts.join(dims.hint("merge"), "dim_id")
smj_join.explain()
smj_join.count()

# In the UI: SQL / DataFrame tab > open each query and compare BroadcastHashJoin vs SortMergeJoin nodes.

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [dim_id#314L, id#312L, label#318]
         +- PhotonBroadcastHashJoin [dim_id#314L], [dim_id#316L], Inner, BuildRight, false, true
            :- PhotonProject [id#312L, (id#312L % 1000) AS dim_id#314L]
            :  +- PhotonRange Range (0, 10000000, step=1, splits=4)
            +- PhotonShuffleExchangeSource
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#1427]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonProject [id#315L AS dim_id#316L, concat(dim_, cast(id#315L as string)) AS label#318]
                        +- PhotonRange Range (0, 1000, step=1, splits=4)


== Photon Explanation ==
The query is fully supported by Photon.
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   Project [dim_id#327L, id#325L, label#331]
   +- SortMergeJoin [dim_i

10000000

In [0]:
# Sample 6: Skewed keys create a few very long tasks
spark.conf.set("spark.job.description", "Sample 6 - data skew")

# Disable AQE skew handling so the skew is clearly visible in the UI
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")

skewed = spark.range(0, 20_000_000).withColumn(
    "key", F.when(F.col("id") % 100 < 90, F.lit(0)).otherwise(F.col("id") % 50)  # ~90% of rows share key 0
)
keys = spark.range(0, 50).withColumnRenamed("id", "key")

skewed.join(keys.hint("merge"), "key").groupBy("key").count().collect()

# In the UI: Stages tab > open the join stage > Summary Metrics.
# A large gap between the Median and Max task duration / shuffle read size indicates skew.

spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")  # Restore default

In [0]:
# Sample 7: AQE coalesces shuffle partitions at runtime
spark.conf.set("spark.job.description", "Sample 7 - AQE")

agg_df = spark.range(0, 10_000_000).groupBy((F.col("id") % 10).alias("k")).count()

spark.conf.set("spark.sql.adaptive.enabled", "false")
agg_df.collect()  # Uses the full spark.sql.shuffle.partitions number of tasks

spark.conf.set("spark.sql.adaptive.enabled", "true")
agg_df.collect()  # AQE coalesces small partitions into fewer tasks

print("spark.sql.shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))
# In the UI: SQL / DataFrame tab > compare both queries. The AQE run shows AdaptiveSparkPlan
# and an AQEShuffleRead node with 'coalesced' partitions, and fewer tasks in the Stages tab.

spark.sql.shuffle.partitions = 200


In [0]:
# Sample 8: Few, large shuffle partitions can cause spill to disk
spark.conf.set("spark.job.description", "Sample 8 - shuffle spill")

original_partitions = spark.conf.get("spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "2")  # Deliberately too few partitions

wide_rows = spark.range(0, 30_000_000).withColumn("payload", F.sha2(F.col("id").cast("string"), 256))
wide_rows.orderBy("payload").write.format("noop").mode("overwrite").save()  # Runs the full sort without storing output

# In the UI: Stages tab > look for 'Spill (Memory)' and 'Spill (Disk)' columns in the sort stage.

# Restore settings
spark.conf.set("spark.sql.shuffle.partitions", original_partitions)
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [0]:
# Sample 9: Values shown in the Environment and Executors tabs
from pyspark.sql import functions as F

spark.conf.unset("spark.job.description")
spark.conf.unset("spark.job.group.id")  # Clear the job group

print("Spark version:       ", spark.version)
print("Default parallelism: ", spark.range(0, 1000).select(F.spark_partition_id()).distinct().count())  # Roughly total executor cores
print("Shuffle partitions:  ", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled:         ", spark.conf.get("spark.sql.adaptive.enabled"))
print("Spark UI URL:        ", f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/#setting/clusters/{spark.conf.get('spark.databricks.clusterUsageTags.clusterId')}/sparkUi")

# In the UI: Environment tab lists all Spark properties; Executors tab shows cores,
# memory, task counts, GC time and shuffle read/write per executor.

Spark version:        4.1.0
Default parallelism:  4
Shuffle partitions:   200
AQE enabled:          true
Spark UI URL:         https://adb-7405605667720717.17.azuredatabricks.net/#setting/clusters/1123-022042-axca9fhh/sparkUi
